# Nutrition Label OCR - Data Collection Notebook

**Objective:** Download, augment, and organize nutrition label datasets from Kaggle

**Sections:**
1. Setup & Dependencies
2. Kaggle Dataset Download
3. Data Exploration
4. Image Download
5. Data Augmentation
6. Dataset Organization & Reports


## 1. Setup & Dependencies

Install and import required libraries.


In [1]:
# Install dependencies (run once)
# !pip install -r ../requirements.txt


In [2]:
# Imports
import os
import sys
import pandas as pd
import numpy as np
from PIL import Image as PILImage
from pathlib import Path
from tqdm.notebook import tqdm
import logging
import json
from datetime import datetime

# Add src to path
sys.path.append(str(Path.cwd().parent))

from src.utils.data_collection import (
    validate_image,
    download_image,
    apply_augmentation,
    translate_nutrients
)

print("✓ All imports successful")


✓ All imports successful


/Users/glarruda/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
# Configure logging
log_dir = Path('../logs')
log_dir.mkdir(exist_ok=True)

log_file = log_dir / f'data_collection_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("Data collection notebook started")

print(f"✓ Logging configured: {log_file}")


INFO:__main__:Data collection notebook started


✓ Logging configured: ../logs/data_collection_20260405_234121.log


In [4]:
# Configuration
CONFIG = {
    'datasets': [
        'mariogemoll/nutrition-facts',
        'shensivam/nutritional-facts-from-food-label',
        'gheysar4real/iranian-nutritional-fact-label'
    ],
    'data_dir': Path('../data'),
    'raw_dir': Path('../data/raw'),
    'processed_dir': Path('../data/processed'),
    'images_original_dir': Path('../data/images/original'),
    'images_augmented_dir': Path('../data/images/augmented'),
    'images_rejected_dir': Path('../data/images/rejected'),
    'min_image_size': (200, 200),
    'augmentation_count': 3,
    'download_timeout': 30,
    'translation_target': 'pt'
}

# Verify directories exist
for dir_path in [CONFIG['raw_dir'], CONFIG['processed_dir'], 
                 CONFIG['images_original_dir'], CONFIG['images_augmented_dir'],
                 CONFIG['images_rejected_dir']]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("✓ Configuration loaded")
print(f"  Datasets: {len(CONFIG['datasets'])}")
print(f"  Data directory: {CONFIG['data_dir']}")


✓ Configuration loaded
  Datasets: 3
  Data directory: ../data


## 2. Kaggle Dataset Download

Download dos datasets definidos em `CONFIG['datasets']` usando a Kaggle API.

**Pré-requisitos da Kaggle API:**
- Instalar o pacote `kaggle` (já listado no `requirements.txt`)
- Criar token em Kaggle > Account > API > *Create New Token*
- Salvar `kaggle.json` em `~/.kaggle/kaggle.json`
- Ajustar permissões do arquivo: `chmod 600 ~/.kaggle/kaggle.json`

> ⚠️ Esta seção depende de credenciais locais e não deve ser executada automaticamente no repositório.


In [ ]:
# Kaggle API authentication
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

print("✓ Kaggle API authentication successful")


In [ ]:
# Download datasets from Kaggle
download_results = []

for dataset in CONFIG['datasets']:
    logger.info(f"Starting download: {dataset}")

    try:
        api.dataset_download_files(
            dataset,
            path=str(CONFIG['raw_dir']),
            unzip=True,
            quiet=False
        )

        logger.info(f"✓ Download completed: {dataset}")
        download_results.append({
            'dataset': dataset,
            'status': 'success',
            'error': None
        })

    except Exception as e:
        logger.error(f"✗ Download failed for {dataset}: {e}")
        download_results.append({
            'dataset': dataset,
            'status': 'failed',
            'error': str(e)
        })

download_summary_df = pd.DataFrame(download_results)
display(download_summary_df)

success_count = (download_summary_df['status'] == 'success').sum()
failure_count = (download_summary_df['status'] == 'failed').sum()
print(f"\nDownload summary: {success_count} succeeded, {failure_count} failed")


In [ ]:
# List downloaded CSV files
downloaded_csvs = sorted(CONFIG['raw_dir'].rglob('*.csv'))

print(f"Found {len(downloaded_csvs)} CSV files in {CONFIG['raw_dir']}")
for csv_file in downloaded_csvs:
    print(f" - {csv_file.relative_to(CONFIG['raw_dir'])}")


## 3. Data Exploration

Load and analyze the downloaded datasets:
- Row/column counts
- Language distribution
- Nutrient completeness
- Image availability


In [ ]:
# Load all CSV files
datasets = {}

raw_files = downloaded_csvs if 'downloaded_csvs' in globals() else sorted(CONFIG['raw_dir'].rglob('*.csv'))

for csv_file in raw_files:
    dataset_key = csv_file.relative_to(CONFIG['raw_dir']).as_posix()
    try:
        df = pd.read_csv(csv_file)
        datasets[dataset_key] = df
        logger.info(f"Loaded {dataset_key}: {len(df)} rows, {len(df.columns)} columns")
    except Exception as e:
        logger.error(f"Failed to load {csv_file}: {e}")

print(f"✓ Loaded {len(datasets)} datasets")


In [ ]:
# Exploration: Dataset sizes
print("=== Dataset Sizes ===")
for name, df in datasets.items():
    print(f"{name}:")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Columns: {list(df.columns[:10])}...")
    print()


In [ ]:
# Exploration: Detect image columns
print("=== Image URL/Path Columns ===")

image_columns = {}

for name, df in datasets.items():
    # Look for columns containing 'url', 'image', 'path', 'link'
    potential_img_cols = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in ['url', 'image', 'path', 'link', 'img'])
    ]

    image_columns[name] = potential_img_cols
    print(f"{name}: {potential_img_cols}")


In [ ]:
# Exploration: Language detection
print("\n=== Language Distribution ===")

for name, df in datasets.items():
    # Check for explicit language column
    lang_col = None
    for col in df.columns:
        if 'lang' in col.lower() or 'language' in col.lower():
            lang_col = col
            break

    if lang_col:
        print(f"{name}:")
        print(df[lang_col].value_counts())
    else:
        detected_lang = 'unknown (requires explicit metadata)'
        print(f"{name}: {detected_lang}")
    print()


In [ ]:
# Exploration: Nutrient columns
print("=== Nutrient Information Columns ===")

for name, df in datasets.items():
    # Look for numeric columns (likely nutrients)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    print(f"{name}:")
    print(f"  Numeric columns: {len(numeric_cols)}")
    print(f"  Sample: {numeric_cols[:10]}")

    # Check completeness
    if numeric_cols:
        completeness = df[numeric_cols].notna().mean() * 100
        avg_completeness = completeness.mean()
        print(f"  Avg completeness: {avg_completeness:.1f}%")
    print()


In [ ]:
# Summary statistics
total_rows = sum(len(df) for df in datasets.values())
total_datasets = len(datasets)

summary = {
    'Total datasets': total_datasets,
    'Total rows': total_rows,
    'Estimated images': total_rows  # Will be refined after download
}

print("\n=== Exploration Summary ===")
for key, value in summary.items():
    print(f"{key}: {value}")


## 4. Image Download

Download images referenced in the datasets:
- Extract image URLs/paths
- Download with validation
- Track successes/failures


In [ ]:
import io
import re

# Prepare download tracking
download_tracking = []


In [ ]:
# Download images from each dataset
for dataset_name, df in datasets.items():
    logger.info(f"Processing images from {dataset_name}")

    safe_dataset_name = re.sub(r'[^A-Za-z0-9._-]+', '_', str(dataset_name)).strip('._-') or 'dataset'

    # Get image column
    img_cols = image_columns.get(dataset_name, [])

    if not img_cols:
        logger.warning(f"No image columns found for {dataset_name}, skipping")
        continue

    # Use first image column
    img_col = img_cols[0]

    # Filter rows with valid URLs
    urls = df[img_col].dropna().tolist()

    print(f"\n{dataset_name}: {len(urls)} images to download")

    for idx, url in enumerate(tqdm(urls, desc=f"Downloading {dataset_name}")):
        image_id = f"{safe_dataset_name}_{idx:04d}"

        # Download
        download_result = download_image(url, timeout=CONFIG['download_timeout'])

        if not download_result['success']:
            download_tracking.append({
                'image_id': image_id,
                'dataset': dataset_name,
                'url': url,
                'status': 'download_failed',
                'reason': download_result['error']
            })
            continue

        # Validate
        img_bytes = io.BytesIO(download_result['data'])
        validation = validate_image(
            img_bytes,
            min_width=CONFIG['min_image_size'][0],
            min_height=CONFIG['min_image_size'][1]
        )

        if not validation['valid']:
            # Move to rejected
            rejected_path = CONFIG['images_rejected_dir'] / f"{image_id}.jpg"
            with open(rejected_path, 'wb') as f:
                f.write(download_result['data'])

            download_tracking.append({
                'image_id': image_id,
                'dataset': dataset_name,
                'url': url,
                'status': 'rejected',
                'reason': validation['reason']
            })
            continue

        # Save valid image
        img_path = CONFIG['images_original_dir'] / f"{image_id}.jpg"
        with open(img_path, 'wb') as f:
            f.write(download_result['data'])

        download_tracking.append({
            'image_id': image_id,
            'dataset': dataset_name,
            'url': url,
            'status': 'success',
            'width': validation['width'],
            'height': validation['height'],
            'format': validation['format'],
            'path': str(img_path.relative_to(CONFIG['data_dir']))
        })


In [ ]:
# Download summary
tracking_columns = ['image_id', 'dataset', 'url', 'status', 'reason', 'width', 'height', 'format', 'path']
tracking_df = pd.DataFrame(download_tracking)

if tracking_df.empty:
    tracking_df = pd.DataFrame(columns=tracking_columns)

print("\n=== Download Summary ===")
if tracking_df.empty:
    print("No downloads processed.")
else:
    print(tracking_df['status'].value_counts())

# Save tracking
tracking_df.to_csv(CONFIG['processed_dir'] / 'download_tracking.csv', index=False)
print(f"\n✓ Tracking saved to {CONFIG['processed_dir'] / 'download_tracking.csv'}")


In [ ]:
# Success rate by dataset
print("\n=== Success Rate by Dataset ===")
if tracking_df.empty:
    print("No tracking data available.")
else:
    success_by_dataset = tracking_df.groupby('dataset')['status'].apply(
        lambda x: (x == 'success').sum() / len(x) * 100
    )
    print(success_by_dataset)


## 5. Data Augmentation

Apply augmentation to expand the dataset:
- Rotation, brightness, blur, crop
- 3-5 variations per image
- Target: ~4x original size


In [ ]:
# Get successfully downloaded images
successful_images = tracking_df[tracking_df['status'] == 'success']

print(f"Images to augment: {len(successful_images)}")
print(f"Target augmented images: {len(successful_images) * CONFIG['augmentation_count']}")


In [ ]:
# Apply augmentation
augmentation_tracking = []
augmentation_errors = []

for idx, row in tqdm(successful_images.iterrows(), 
                     total=len(successful_images),
                     desc="Augmenting images"):
    
    try:
        # Load original image
        img_path = CONFIG['data_dir'] / row['path']
        with PILImage.open(img_path) as original_img:
            # Apply augmentation
            augmented_results = apply_augmentation(
                original_img,
                num_variations=CONFIG['augmentation_count'],
                seed=idx  # Reproducible
            )
        
        # Save augmented images
        for aug_idx, aug_result in enumerate(augmented_results):
            aug_image_id = f"{row['image_id']}_aug{aug_idx:02d}"
            aug_path = CONFIG['images_augmented_dir'] / f"{aug_image_id}.jpg"
            
            aug_result['image'].save(aug_path, 'JPEG')
            
            augmentation_tracking.append({
                'image_id': aug_image_id,
                'original_image_id': row['image_id'],
                'dataset': row['dataset'],
                'augmentation_type': aug_result['augmentation_type'],
                'path': str(aug_path.relative_to(CONFIG['data_dir']))
            })
    except Exception as error:
        augmentation_errors.append({
            'image_id': row['image_id'],
            'dataset': row['dataset'],
            'path': row['path'],
            'error': str(error)
        })
        print(f"⚠️ Failed to augment {row['image_id']}: {error}")
        continue


In [ ]:
# Augmentation summary
aug_df = pd.DataFrame(augmentation_tracking)
successful_count = len(successful_images)
expansion_factor = (len(aug_df) / successful_count) if successful_count > 0 else 0

print("\n=== Augmentation Summary ===")
print(f"Original images: {successful_count}")
print(f"Augmented images: {len(aug_df)}")
print(f"Total images: {successful_count + len(aug_df)}")
print(f"Expansion factor: {expansion_factor:.1f}x")

# Save tracking
aug_df.to_csv(CONFIG['processed_dir'] / 'augmentation_log.csv', index=False)
print(f"\n✓ Augmentation log saved")

if augmentation_errors:
    errors_df = pd.DataFrame(augmentation_errors)
    errors_path = CONFIG['processed_dir'] / 'augmentation_errors.csv'
    errors_df.to_csv(errors_path, index=False)
    print(f"Augmentation errors: {len(augmentation_errors)}")
    print(f"⚠️ Augmentation errors log saved: {errors_path}")


In [ ]:
# Most common augmentation types
print("\n=== Augmentation Types Distribution ===")
if not aug_df.empty and 'augmentation_type' in aug_df.columns:
    print(aug_df['augmentation_type'].value_counts().head(10))
else:
    print("No augmentation distribution available (empty dataframe or missing column).")


## 6. Dataset Organization & Translation

Final steps:
- Translate nutrient names to Portuguese
- Consolidate all data into single CSV
- Generate final reports


In [ ]:
# Prepare consolidated dataset with translated nutrients (PT)
NUTRIENT_TRANSLATIONS_PT = {
    'calories': 'calorias',
    'energy': 'energia',
    'protein': 'proteína',
    'proteins': 'proteínas',
    'carbohydrate': 'carboidrato',
    'carbohydrates': 'carboidratos',
    'total_fat': 'gordura_total',
    'fat': 'gordura',
    'total_fats': 'gorduras_totais',
    'saturated_fat': 'gordura_saturada',
    'trans_fat': 'gordura_trans',
    'fiber': 'fibra',
    'dietary_fiber': 'fibra_alimentar',
    'sugar': 'açúcar',
    'sugars': 'açúcares',
    'sodium': 'sódio',
    'salt': 'sal',
    'cholesterol': 'colesterol'
}

def normalize_nutrients(row_series, nutrient_columns):
    nutrients = {}
    if row_series is None:
        return nutrients

    for nutrient_name in nutrient_columns:
        value = row_series.get(nutrient_name)
        if pd.notna(value):
            nutrients[str(nutrient_name)] = float(value)

    return nutrients

def translate_nutrient_keys(nutrients, target_lang='pt'):
    if target_lang != 'pt':
        return nutrients.copy()

    translated = {}
    for nutrient_name, value in nutrients.items():
        lookup_key = str(nutrient_name).strip().lower()
        translated_key = NUTRIENT_TRANSLATIONS_PT.get(lookup_key, str(nutrient_name))
        translated[translated_key] = value

    return translated

def serialize_translated_nutrients(nutrients):
    translated = translate_nutrient_keys(nutrients, target_lang=CONFIG['translation_target'])
    return json.dumps(translated, ensure_ascii=False, sort_keys=True)

consolidated_rows = []
original_nutrients_by_image = {}

# Add original images
for _, row in successful_images.iterrows():
    dataset_name = row['dataset']
    original_df = datasets[dataset_name]
    nutrient_cols = original_df.select_dtypes(include=[np.number]).columns.tolist()

    original_match = original_df[original_df['image_url'] == row['url']] if 'image_url' in original_df.columns else pd.DataFrame()
    source_row = original_match.iloc[0] if not original_match.empty else (original_df.iloc[0] if len(original_df) > 0 else None)
    nutrients_original = normalize_nutrients(source_row, nutrient_cols)
    original_nutrients_by_image[row['image_id']] = nutrients_original

    consolidated_rows.append({
        'image_id': row['image_id'],
        'source_dataset': dataset_name,
        'image_path': row['path'],
        'is_augmented': False,
        'augmentation_type': None,
        'language': CONFIG['translation_target'],
        'width': row['width'],
        'height': row['height'],
        'nutrients_json': serialize_translated_nutrients(nutrients_original)
    })

# Add augmented images with inherited nutrients from original image
for _, row in aug_df.iterrows():
    inherited_nutrients = original_nutrients_by_image.get(row.get('original_image_id'), {}).copy()

    consolidated_rows.append({
        'image_id': row['image_id'],
        'source_dataset': row['dataset'],
        'image_path': row['path'],
        'is_augmented': True,
        'augmentation_type': row['augmentation_type'],
        'language': CONFIG['translation_target'],
        'width': None,
        'height': None,
        'nutrients_json': serialize_translated_nutrients(inherited_nutrients)
    })

consolidated_df = pd.DataFrame(consolidated_rows)
print(f"✓ Consolidated dataset: {len(consolidated_df)} rows")

if not consolidated_df.empty:
    empty_nutrients = consolidated_df['nutrients_json'].isna().sum()
    print(f"  Rows with missing nutrients_json: {empty_nutrients}")


In [ ]:
# Validate nutrient translation in consolidated dataset
print("Validating translated nutrients in consolidated dataset...")

if not consolidated_df.empty:
    sample_payload = json.loads(consolidated_df.iloc[0]['nutrients_json'])
    print("
✓ Example translated nutrients:")
    for idx, (nutrient, value) in enumerate(sample_payload.items()):
        if idx >= 5:
            break
        print(f"  {nutrient}: {value}")

    total_rows = len(consolidated_df)
    translated_rows = 0
    for payload in consolidated_df['nutrients_json']:
        nutrients = json.loads(payload)
        if any(k in NUTRIENT_TRANSLATIONS_PT.values() for k in nutrients.keys()):
            translated_rows += 1

    print(f"
Rows with translated nutrient keys: {translated_rows}/{total_rows}")
else:
    print("No rows available for translation validation.")


In [ ]:
# Save consolidated dataset
output_csv = CONFIG['processed_dir'] / 'consolidated_dataset.csv'
consolidated_df.to_csv(output_csv, index=False)

print(f"✓ Saved consolidated dataset: {output_csv}")
print(f"  Total images: {len(consolidated_df)}")
print(f"  Original: {(~consolidated_df['is_augmented']).sum()}")
print(f"  Augmented: {consolidated_df['is_augmented'].sum()}")


## Final Report


In [ ]:
# Generate final report
print("=" * 60)
print("NUTRITION LABEL OCR - DATA COLLECTION REPORT")
print("=" * 60)
print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n--- DATASETS ---")
print(f"Source datasets: {len(CONFIG['datasets'])}")
for ds in CONFIG['datasets']:
    print(f"  - {ds}")

print("\n--- DOWNLOADS ---")
print(f"Total download attempts: {len(tracking_df)}")
print(f"Successful: {(tracking_df['status'] == 'success').sum()}")
print(f"Rejected: {(tracking_df['status'] == 'rejected').sum()}")
print(f"Failed: {(tracking_df['status'] == 'download_failed').sum()}")
success_rate = ((tracking_df['status'] == 'success').sum() / len(tracking_df) * 100) if len(tracking_df) else 0
print(f"Success rate: {success_rate:.1f}%")

print("\n--- AUGMENTATION ---")
print(f"Original images: {len(successful_images)}")
print(f"Augmented images: {len(aug_df)}")
print(f"Total images: {len(consolidated_df)}")
expansion_factor = (len(aug_df) / len(successful_images)) if len(successful_images) else 0
print(f"Expansion factor: {expansion_factor:.2f}x")

print("\n--- STORAGE ---")
original_dir_size = sum(f.stat().st_size for f in CONFIG['images_original_dir'].glob('*.jpg'))
augmented_dir_size = sum(f.stat().st_size for f in CONFIG['images_augmented_dir'].glob('*.jpg'))
total_size = (original_dir_size + augmented_dir_size) / (1024**3)

print(f"Original images: {original_dir_size / (1024**2):.1f} MB")
print(f"Augmented images: {augmented_dir_size / (1024**2):.1f} MB")
print(f"Total disk usage: {total_size:.2f} GB")

print("\n--- OUTPUT FILES ---")
print(f"Consolidated CSV: {output_csv}")
print(f"Download tracking: {CONFIG['processed_dir'] / 'download_tracking.csv'}")
print(f"Augmentation log: {CONFIG['processed_dir'] / 'augmentation_log.csv'}")
print(f"Execution log: {log_file}")

print("\n" + "=" * 60)
print("✓ DATA COLLECTION COMPLETE")
print("=" * 60)


In [ ]:
# Save report to file
report_file = CONFIG['processed_dir'] / f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

with open(report_file, 'w') as f:
    f.write("NUTRITION LABEL OCR - DATA COLLECTION REPORT\n")
    f.write("=" * 60 + "\n")
    f.write(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"\nTotal images: {len(consolidated_df)}\n")
    f.write(f"Success rate: {success_rate:.1f}%\n")
    f.write(f"Total disk usage: {total_size:.2f} GB\n")

print(f"✓ Report saved: {report_file}")
